In [6]:
# ================== 0. Setup & imports ==================
from google.colab import files
uploaded = files.upload()  # upload data_de_vi.csv from your PC

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor  # pip install xgboost if needed

Saving data_de_vi.csv to data_de_vi (2).csv


In [7]:
# ================== 1. Load data ==================
df = pd.read_csv("data_de_vi.csv")  # make sure this name matches your file

X    = df[["temp", "loading", "conc"]]
y_th = df["density"]
y_sp = df["visc"]

X_train, X_test, y_th_train, y_th_test = train_test_split(
    X, y_th, test_size=0.2, random_state=42
)
X_train2, X_test2, y_sp_train, y_sp_test = train_test_split(
    X, y_sp, test_size=0.2, random_state=42
)

# ================== 2. Helpers ==================
def metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

def print_metrics(name, target, y_true, y_pred):
    r2, mae, rmse = metrics(y_true, y_pred)
    print(f"{name:22s} - {target}: R^2={r2:.4f}, MAE={mae:.4f}, RMSE={rmse:.4f}")

# formulas for linear / ridge
def print_linear_like_formula(model, target_name, label):
    coef = model.coef_
    inter = model.intercept_
    print(f"\n{label} formula for {target_name}:")
    print(
        f"{target_name} = {inter:.6f}"
        f" + ({coef[0]:.6f})*temp"
        f" + ({coef[1]:.6f})*loading"
        f" + ({coef[2]:.6f})*conc"
    )

# formula for polynomial
def print_poly_formula(pipe, target_name, degree):
    poly_step = pipe.named_steps["poly"]
    lin_step  = pipe.named_steps["lin"]
    names = poly_step.get_feature_names_out(["temp", "loading", "conc"])
    coefs = lin_step.coef_
    inter = lin_step.intercept_
    print(f"\nPolynomial (deg={degree}) formula for {target_name}:")
    print(f"{target_name} = {inter:.6f}", end="")
    # names[0] is "1" (bias term), skip it
    for name, c in zip(names[1:], coefs[1:]):
        print(f" + ({c:.6f})*{name}", end="")
    print()

# model list for generic training
def make_models():
    poly_degree = 2
    return [
        ("Linear Regression",
         LinearRegression()),
        (f"Polynomial Regression (deg={poly_degree})",
         Pipeline([
             ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
             ("lin", LinearRegression())
         ])),
        ("Random Forest",
         RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ("Gradient Boosting",
         GradientBoostingRegressor(
             n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)),
        ("XGBoost",
         XGBRegressor(
             n_estimators=300, learning_rate=0.05, max_depth=4,
             subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1)),
        ("Ridge Regression",
         Ridge(alpha=1.0)),
        ("KNN",
         KNeighborsRegressor(n_neighbors=5, weights="distance")),
        ("Neural Network",
         MLPRegressor(
             hidden_layer_sizes=(64, 64), activation="relu",
             learning_rate_init=0.001, max_iter=500, random_state=42)),
    ]

# ================== 3. Train all 8 models & metrics ==================
print("=== Results for density ===")
models_th = make_models()
for name, model in models_th:
    model.fit(X_train, y_th_train)
    y_pred = model.predict(X_test)
    print_metrics(name, "density", y_th_test, y_pred)

print("\n=== Results for visc ===")
models_sp = make_models()
for name, model in models_sp:
    model.fit(X_train2, y_sp_train)
    y_pred = model.predict(X_test2)
    print_metrics(name, "visc", y_sp_test, y_pred)

# ================== 4. Explicit formulas ==================
# Linear
lin_th = next(m for n, m in models_th if n == "Linear Regression")
lin_sp = next(m for n, m in models_sp if n == "Linear Regression")
print_linear_like_formula(lin_th, "density", "Linear")
print_linear_like_formula(lin_sp, "visc", "Linear")

# Polynomial (degree 2)
poly_name = "Polynomial Regression (deg=2)"
poly_th = next(m for n, m in models_th if n == poly_name)
poly_sp = next(m for n, m in models_sp if n == poly_name)
print_poly_formula(poly_th, "density", degree=2)
print_poly_formula(poly_sp, "visc", degree=2)

# Ridge
ridge_th = next(m for n, m in models_th if n == "Ridge Regression")
ridge_sp = next(m for n, m in models_sp if n == "Ridge Regression")
print_linear_like_formula(ridge_th, "density", "Ridge")
print_linear_like_formula(ridge_sp, "visc", "Ridge")



=== Results for density ===
Linear Regression      - density: R^2=0.9949, MAE=3.0654, RMSE=4.3601
Polynomial Regression (deg=2) - density: R^2=0.9981, MAE=2.0079, RMSE=2.6757
Random Forest          - density: R^2=0.9982, MAE=1.7547, RMSE=2.5982
Gradient Boosting      - density: R^2=0.9672, MAE=6.4759, RMSE=11.0578
XGBoost                - density: R^2=0.9892, MAE=4.0797, RMSE=6.3517
Ridge Regression       - density: R^2=0.9948, MAE=3.0874, RMSE=4.3982
KNN                    - density: R^2=0.8628, MAE=18.5336, RMSE=22.6062
Neural Network         - density: R^2=-3.4878, MAE=104.3188, RMSE=129.3082

=== Results for visc ===
Linear Regression      - visc: R^2=0.9568, MAE=0.0430, RMSE=0.0552
Polynomial Regression (deg=2) - visc: R^2=0.9836, MAE=0.0236, RMSE=0.0341
Random Forest          - visc: R^2=0.9636, MAE=0.0356, RMSE=0.0507
Gradient Boosting      - visc: R^2=0.9594, MAE=0.0250, RMSE=0.0536
XGBoost                - visc: R^2=0.9663, MAE=0.0240, RMSE=0.0488
Ridge Regression       - visc

In [8]:
# ================== 5. Predictions from all models ==================
new_data = pd.DataFrame({
    "temp":   [25.0],
    "loading":[0.3],
    "conc":   [0.8]
})

print("\n=== Predictions for density ===")
for name, model in models_th:
    pred = float(model.predict(new_data)[0])
    print(f"{name:25s} -> density = {pred:.4f}")

print("\n=== Predictions for visc ===")
for name, model in models_sp:
    pred = float(model.predict(new_data)[0])
    print(f"{name:25s} -> visc = {pred:.4f}")



=== Predictions for density ===
Linear Regression         -> density = 558.8545
Polynomial Regression (deg=2) -> density = 544.1191
Random Forest             -> density = 496.8286
Gradient Boosting         -> density = 497.1539
XGBoost                   -> density = 475.5117
Ridge Regression          -> density = 557.7990
KNN                       -> density = 457.3924
Neural Network            -> density = 204.6028

=== Predictions for visc ===
Linear Regression         -> visc = 3.9114
Polynomial Regression (deg=2) -> visc = 4.1112
Random Forest             -> visc = 3.7851
Gradient Boosting         -> visc = 3.7689
XGBoost                   -> visc = 3.6302
Ridge Regression          -> visc = 3.9077
KNN                       -> visc = 3.6465
Neural Network            -> visc = 3.6440
